In [ ]:
##### DATA #################################################################################################

In [ ]:
import pandas as pd
import torch
from transformers import BertModel, BertTokenizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from transformers import BertTokenizer
from kobert_transformers import get_tokenizer
from torch.utils.data import DataLoader, Dataset

In [ ]:
file_path_new = "./data2/Training.xlsx"
df_new = pd.read_excel(file_path_new)

In [ ]:
filtered_df = df_new[['감정_대분류', '사람문장1', '사람문장2', '사람문장3']]
filtered_df = filtered_df.fillna('')

In [ ]:
filtered_df.loc[:, 'combined_sentences'] = filtered_df[['사람문장1', '사람문장2', '사람문장3']].apply(
    lambda row: ' '.join(' '.join(row.values.astype(str)).split()).strip(), axis=1)

In [ ]:
joy = filtered_df[filtered_df['감정_대분류'] == '기쁨']['combined_sentences'].sample(n=200, random_state=42).tolist()
amb = filtered_df[filtered_df['감정_대분류'] == '당황']['combined_sentences'].sample(n=200, random_state=42).tolist()
ang = filtered_df[filtered_df['감정_대분류'] == '분노']['combined_sentences'].sample(n=200, random_state=42).tolist()
anx = filtered_df[filtered_df['감정_대분류'] == '불안']['combined_sentences'].sample(n=200, random_state=42).tolist()
sca = filtered_df[filtered_df['감정_대분류'] == '상처']['combined_sentences'].sample(n=200, random_state=42).tolist()
sad = filtered_df[filtered_df['감정_대분류'] == '슬픔']['combined_sentences'].sample(n=200, random_state=42).tolist()

In [ ]:
combined_list = joy + amb + ang + anx + sca + sad
print(combined_list[0:10])
print(len(combined_list))

In [ ]:
##### KR-BERT #################################################################################################

In [ ]:
tokenizer = BertTokenizer.from_pretrained('snunlp/KR-BERT-char16424')
model = BertModel.from_pretrained('snunlp/KR-BERT-char16424')

In [ ]:
# 위에 있는거랑 같은거임 메모리 문제 
sentences = combined_list
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Custom Dataset 클래스
class SentenceDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_len=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        inputs = self.tokenizer(
            sentence, 
            return_tensors='pt', 
            padding='max_length', 
            truncation=True, 
            max_length=self.max_len
        )
        return {key: value.squeeze(0) for key, value in inputs.items()}

# Dataset 및 DataLoader 생성 (배치 사이즈 조절 가능)
batch_size = 16
dataset = SentenceDataset(sentences, tokenizer)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# 배치 단위로 임베딩 계산
all_embeddings = []
model.eval()  # 추론 모드로 변경 (dropout 비활성화)

with torch.no_grad():  # 그래디언트 계산 비활성화
    for batch in dataloader:
        batch = {key: value.to(device) for key, value in batch.items()}
        outputs = model(**batch)
        # CLS 토큰 임베딩 추출 (문장 임베딩)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

# 임베딩을 하나의 배열로 합치기
all_embeddings = np.vstack(all_embeddings)

In [ ]:
##### PCA #################################################################################################

In [ ]:
n_components = 80  
pca = PCA(n_components=n_components)

# 임베딩에 PCA 적용
pca_embeddings = pca.fit_transform(all_embeddings)

explained_variance = pca.explained_variance_ratio_
print(f'50차원으로 축소한 후 설명 분산 비율: {np.sum(explained_variance)}')
# 80% 이상이면 차원 축소 적절 가능성 큼 

In [ ]:
##### TF-IDF #################################################################################################

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# TF-IDF 벡터화 함수
def get_tfidf_features(sentences, max_features=128):
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(sentences).toarray()
    return tfidf_matrix

# TF-IDF 피처 생성
tfidf_features = get_tfidf_features(sentences, max_features=128)

# BERT 임베딩과 TF-IDF 벡터 결합
combined_embeddings = np.hstack((pca_embeddings, tfidf_features))  
print(f'결합된 임베딩 차원: {combined_embeddings.shape}')

In [ ]:
##### 감정 분석 #################################################################################################

In [ ]:
from transformers import pipeline

# 감정 분석 모델 로드
sentiment_analyzer = pipeline("sentiment-analysis", model="snunlp/KR-BERT-char16424")

# 감정 분석 결과 (긍정, 부정) 추출
sentiment_scores = []
for sentence in sentences:
    result = sentiment_analyzer(sentence)[0]
    # 긍정이면 1, 부정이면 0으로 변환
    score = 1 if result['label'] == 'positive' else 0
    sentiment_scores.append(score)

# 감정 점수를 numpy 배열로 변환
sentiment_scores = np.array(sentiment_scores).reshape(-1, 1)

# BERT + TF-IDF + 감정 점수 결합
final_embeddings = np.hstack((combined_embeddings, sentiment_scores))
print(f'최종 결합된 임베딩 차원: {final_embeddings.shape}')

In [ ]:
new = np.hstack((tfidf_features, sentiment_scores))

In [ ]:
def five_number_summary(matrix):
    flat_values = matrix.flatten()  
    summary = {
        "min": np.min(flat_values),
        "25%": np.percentile(flat_values, 25),
        "median": np.median(flat_values),
        "75%": np.percentile(flat_values, 75),
        "max": np.max(flat_values)
    }
    return summary

In [ ]:
def manhattan_similarity_matrix(embeddings):
    num_sentences = embeddings.shape[0]
    similarity_matrix = np.zeros((num_sentences, num_sentences))

    for i in range(num_sentences):
        for j in range(num_sentences):
            similarity_matrix[i, j] = np.sum(np.abs(embeddings[i] - embeddings[j]))

    return similarity_matrix

In [ ]:
manhattan_matrix = manhattan_similarity_matrix(all_embeddings)

summary = five_number_summary(manhattan_matrix)
for stat, values in summary.items():
    print(f"{stat}: {values}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sentence_embeddings = np.array(all_embeddings)

cosine_similarity = cosine_similarity(sentence_embeddings)

summary = five_number_summary(cosine_similarity)
for stat, values in summary.items():
    print(f"{stat}: {values}")

In [ ]:
x = final_embeddings[:200, :209]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sentence_embeddings = np.array(x)

cosine_similarity = cosine_similarity(sentence_embeddings)

summary = five_number_summary(cosine_similarity)
for stat, values in summary.items():
    print(f"{stat}: {values}")

In [ ]:
np.sum(cosine_similarity >= 0.5)

In [ ]:
cosine_similarity[0:8,400:408]

In [ ]:
cosine_similarity[0:8,0:8]

In [ ]:
combined_list[0:8]

In [ ]:
combined_list[400:408]

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def soft_cosine_similarity(embeddings, word_similarity_matrix):
    num_sentences = embeddings.shape[0]
    scm_matrix = np.zeros((num_sentences, num_sentences))

    for i in range(num_sentences):
        for j in range(num_sentences):
            A = embeddings[i].reshape(1, -1)
            B = embeddings[j].reshape(1, -1)

            # Numerator: A^T S B
            numerator = np.dot(np.dot(A, word_similarity_matrix), B.T)

            # Denominator: sqrt(A^T S A) * sqrt(B^T S B)
            denominator = np.sqrt(np.dot(np.dot(A, word_similarity_matrix), A.T)) * np.sqrt(np.dot(np.dot(B, word_similarity_matrix), B.T))

            # SCM calculation
            scm_matrix[i, j] = (numerator / denominator).item()

    return scm_matrix


In [ ]:
# 코사인 유사도 계산
word_similarity_matrix = cosine_similarity(final_embeddings.T)  # 단어 간 유사도 행렬 생성
scm_matrix = soft_cosine_similarity(final_embeddings, word_similarity_matrix)

# 유사도 행렬 출력
print("Soft Cosine Similarity Matrix:")
print(scm_matrix)

In [ ]:
summary = five_number_summary(scm_matrix)
for stat, values in summary.items():
    print(f"{stat}: {values}")

In [ ]:
scm_matrix[0:10,1000:1010]